# 07 · The best public submission (0.94155)

This is **run 2**, written on 2026-11-15, and the submission with the best public leaderboard score: **0.94155**.
Its frozen code and the submitted file are kept unchanged in `best_0.94155/`; this notebook rebuilds the same model
from the shared code and checks that it gives exactly the same predictions.

**Model.** For each of the 5 image models of notebook 01 (DINOv2, SigLIP, EVA-02, OpenCLIP ViT-H, OpenAI CLIP):
the photo + mirror embedding → **extra trees** (300 randomised decision trees, √features per split, class-balanced,
seed 0). The five models' class probabilities are averaged and the most likely class is the prediction.
**Training data:** only the 66 competition photos — no external photos, no helpers, no augmentation.

**Steps in this notebook**

1. Load the 66 training photos and their features from the 5 image models.
2. Write the model: train one extra-trees classifier per image model, average their probabilities.
3. Test it with the same cross-validation as every other notebook.
4. Train it on all 66 photos, predict the 36 test photos, save `submission_best_public.csv`.
5. Check that the file is the same as the file that was submitted, and compare it with the final model.

In [1]:
import numpy as np, pandas as pd
import common as c
from sklearn.metrics import f1_score
pd.set_option('display.width', 200)

## 1. Data and features

In [2]:
train = c.training_set()          # the 66 labelled training photos (no extra photos)
y = train.y.values                # class number 0-3 for each photo

# For each image model: one row of numbers per photo (photo and mirror image averaged).
features = {}
for model_name in c.BACKBONES:    # dinov2_l, siglip_so400m, eva02_l, vith14_laion, vitl14_openai
    features[model_name] = c.embed(train.id, model_name)
    print(f"{model_name:15s} {features[model_name].shape[1]} numbers per photo")

dinov2_l        2048 numbers per photo
siglip_so400m   1152 numbers per photo
eva02_l         768 numbers per photo
vith14_laion    1024 numbers per photo
vitl14_openai   768 numbers per photo


## 2. The model

One extra-trees classifier (300 trees, seed 0) is trained for each image model. To predict a photo, each of the five
classifiers gives a probability for each of the 4 classes; the five are averaged, and the class with the highest
average is the prediction.

In [3]:
def train_five(rows):
    """Train one extra-trees classifier per image model on the training photos in `rows`."""
    return {m: c.MODELS["extra trees"]().fit(features[m][rows], y[rows]) for m in c.BACKBONES}


def predict_five(classifiers, features_to_predict):
    """Average the 5 classifiers' class probabilities and return the most likely class for each photo."""
    probabilities = [classifiers[m].predict_proba(features_to_predict[m]) for m in c.BACKBONES]
    return np.mean(probabilities, axis=0).argmax(axis=1)

## 3. Cross-validation

10 repeats × 5 folds, the same splits as in every other notebook. In each fold the five classifiers are trained on
4/5 of the photos and predict the other 1/5. After 5 folds every photo has one prediction, so each repeat gives one
macro F1 score.

In [4]:
predictions = np.zeros((c.REPEATS, len(train)), dtype=int)   # one row of predictions per repeat

for split_number, (train_rows, held_out_rows) in enumerate(c.splits(train)):
    repeat = split_number // 5                               # splits 0-4 are repeat 0, 5-9 repeat 1, ...
    classifiers = train_five(train_rows)
    held_out = {m: features[m][held_out_rows] for m in c.BACKBONES}
    predictions[repeat, held_out_rows] = predict_five(classifiers, held_out)

scores = [f1_score(y, p, average="macro") for p in predictions]
print(f"CV macro F1 {np.mean(scores):.3f} ± {np.std(scores):.3f}")

dishes, trash = y < 2, y >= 2                                # classes 0-1 are dishes, 2-3 are trash
print(f"dishes F1 {np.mean([f1_score(y[dishes], p[dishes], labels=[0, 1], average='macro') for p in predictions]):.3f}",
      f"  trash F1 {np.mean([f1_score(y[trash], p[trash], labels=[2, 3], average='macro') for p in predictions]):.3f}")
print("photos wrong (times out of 10):", c.wrong_counts(train, predictions))

CV macro F1 0.821 ± 0.029
dishes F1 0.811   trash F1 0.831
photos wrong (times out of 10): {'#2': 8, '#12': 8, '#17': 1, '#19': 8, '#22': 2, '#26': 10, '#31': 10, '#41': 10, '#46': 6, '#47': 1, '#48': 1, '#60': 9, '#61': 4, '#63': 1, '#64': 1, '#66': 1, '#77': 10, '#87': 7, '#99': 6, '#101': 8}


## 4. Train on all 66 photos and predict the 36 test photos

In [5]:
_, test = c.competition()
test_features = {m: c.embed(test.id, m) for m in c.BACKBONES}

classifiers = train_five(np.arange(len(train)))              # all 66 photos
test_classes = predict_five(classifiers, test_features)

submission = pd.DataFrame({"id": test.id, "label": [c.CLASSES[k] for k in test_classes]})
submission.to_csv("submission_best_public.csv", index=False)
submission.label.value_counts()

label
Taking Out Trash|Incomplete    12
Washing Dishes|Incomplete       9
Washing Dishes|Complete         9
Taking Out Trash|Complete       6
Name: count, dtype: int64

## 5. Checks

Is the new file the same as the file that was submitted as run 2?

In [6]:
submitted = pd.read_csv("best_0.94155/submission_public0.94155_extratrees.csv")
both = submission.merge(submitted, on="id", suffixes=("_new", "_submitted"))
print(f"same label on {(both.label_new == both.label_submitted).sum()} of {len(both)} test photos")

same label on 36 of 36 test photos


Where does it differ from the final model (notebook 06, `submission_final.csv`)?

In [7]:
final = pd.read_csv("submission_final.csv")
both = submission.merge(final, on="id", suffixes=("_this_model", "_final_model"))
different = both[both.label_this_model != both.label_final_model]
print(f"same label on {len(both) - len(different)} of {len(both)} test photos")
different

same label on 32 of 36 test photos


,id,label_this_model,label_final_model
5,14,Washing Dishes|Complete,Washing Dishes|Incomplete
6,21,Taking Out Trash|Incomplete,Taking Out Trash|Complete
31,86,Taking Out Trash|Incomplete,Taking Out Trash|Complete
34,95,Taking Out Trash|Incomplete,Washing Dishes|Incomplete


**Why "accidental".** Its cross-validated score (about 0.82) is well below the final model's 0.912, yet it scored
best on the public leaderboard. The public leaderboard scores only part of the 36 test photos (about half), so one
photo more or less changes the score by several points: 0.94155 and the leader's 0.94444 both correspond to a single
wrong photo, and 0.88311 to two. The public score therefore mostly reflects which few photos happen to be in the
public part, not which model is better; the model choice follows the cross-validated score. The test photos were not
looked at or labelled.